# 02 – Xây dựng và đánh giá mô hình học máy (Sleep DSS)

**Bài toán:** phân loại nhị phân. Từ thói quen dùng điện thoại buổi tối và thông tin ngữ cảnh, ước lượng xác suất một người bị **nợ ngủ đáng kể** (`Moderate Debt` hoặc `Severe Sleep Debt` = 1; `Optimal Recovery`, `Mild Deficit` = 0).

Notebook đi qua toàn bộ quy trình:

1. Nạp dữ liệu và chia train/test
2. Tiền xử lý
3. Baseline
4. Huấn luyện và tinh chỉnh 6 mô hình (Logistic Regression, KNN, Decision Tree, Random Forest, XGBoost, ANN) bằng cross-validation 5-fold
5. So sánh trên cross-validation
6. Đánh giá trên tập kiểm tra
7. Chọn mô hình triển khai
8. Thí nghiệm rò rỉ dữ liệu
9. Giải thích mô hình
10. Phân tích ngưỡng
11. Phân tích lỗi
12. Hiệu chỉnh xác suất
13. **Chạy mô hình trên dữ liệu mới**
14. Lưu mô hình

Cột đặc trưng, nhãn, cách chia dữ liệu lấy từ `src/preprocess.py`; danh sách mô hình và lưới tham số lấy từ `src/train.py`. Notebook và script dùng chung một nguồn nên kết quả luôn khớp nhau.

> Chạy **Run All** từ thư mục `notebooks/`. Bước 4 mất vài phút. Notebook **không ghi đè** `models/model.joblib` trừ khi đặt `SAVE_MODEL = True` ở bước 14.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent          # notebook nằm trong notebooks/
sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibrationDisplay
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate

from src.preprocess import (
    CONTEXT_VARS, DECISION_VARS, FEATURES, LABEL_COL, LEAKAGE_COLS, TARGET,
    build_preprocessor, load_dataset, prepare_data, profile_to_frame,
)
from src.train import (
    DEPLOY_MODEL, RANDOM_STATE, SCORING,
    _evaluate, candidate_specs, error_analysis, feature_importance,
    make_pipeline, threshold_analysis,
)
from src.decision import rank_plans, recommend, risk_level

%matplotlib inline
warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.precision", 4)
plt.rcParams["figure.dpi"] = 110

## 1. Nạp dữ liệu và chia train/test

Dữ liệu: *Sleep Debt & Screen Time: Late Night Phone Habits* (Kaggle), đã làm sạch bằng `load_dataset()`. Chia 80% huấn luyện, 20% kiểm tra, **phân tầng theo nhãn** để hai phần có cùng tỉ lệ nợ ngủ. Tập kiểm tra chỉ được dùng **một lần** ở bước 6.

In [ ]:
df = load_dataset()
print(f"Số dòng: {len(df)} | số đặc trưng đầu vào: {len(FEATURES)}")
print(f"Biến quyết định ({len(DECISION_VARS)}): {DECISION_VARS}")
print(f"Biến ngữ cảnh ({len(CONTEXT_VARS)}): {CONTEXT_VARS}")
print(f"Cột bị loại vì rò rỉ ({len(LEAKAGE_COLS)}): {LEAKAGE_COLS}")

display(df[LABEL_COL].value_counts().rename("số dòng").to_frame())

X_train, X_test, y_train, y_test = prepare_data(random_state=RANDOM_STATE)
pd.DataFrame({
    "số dòng": [len(X_train), len(X_test)],
    "tỉ lệ nhãn 1": [y_train.mean(), y_test.mean()],
}, index=["train", "test"])

## 2. Tiền xử lý

Bộ tiền xử lý nằm **bên trong Pipeline**, nên khi cross-validation nó chỉ học (fit) trên phần huấn luyện của từng fold, không nhìn thấy dữ liệu kiểm tra.

| Nhóm cột | Cách xử lý |
| --- | --- |
| Biến số (phút, độ sáng, caffeine, vận động, tuổi) | Chuẩn hóa `StandardScaler` |
| Bộ lọc ánh sáng xanh (0/1) | Giữ nguyên |
| Ứng dụng, giới tính, nghề nghiệp | One-hot, giá trị lạ bị bỏ qua |
| Nhịp sinh học | Mã hóa thứ tự: Morning Lark < Intermediate < Night Owl |

In [ ]:
prep = build_preprocessor().fit(X_train)
X_train_t = prep.transform(X_train)
print(f"{len(FEATURES)} cột thô -> {X_train_t.shape[1]} cột sau mã hóa")
X_train_t.head()

## 3. Baseline

Mô hình mốc luôn đoán lớp phổ biến nhất (nợ ngủ = 1). Mọi mô hình thật phải vượt xa mốc này; ROC-AUC của baseline là 0,5 (không phân biệt được gì).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fitted = {}
cv_rows = []

baseline = make_pipeline(DummyClassifier(strategy="most_frequent"))
scores = cross_validate(baseline, X_train, y_train, cv=cv, scoring=SCORING)
baseline.fit(X_train, y_train)
fitted["Baseline"] = baseline
cv_rows.append({"model": "Baseline",
                **{f"{m}_mean": scores[f"test_{m}"].mean() for m in SCORING},
                **{f"{m}_std": scores[f"test_{m}"].std() for m in SCORING}})
pd.DataFrame(cv_rows)[["model", "accuracy_mean", "recall_mean", "f1_mean", "roc_auc_mean"]]

## 4. Huấn luyện và tinh chỉnh các mô hình

Mỗi mô hình được tinh chỉnh bằng `GridSearchCV` với cross-validation 5-fold **chỉ trên tập huấn luyện**, chọn bộ tham số có ROC-AUC trung bình cao nhất.

| Mô hình | Ý tưởng | Vì sao thử |
| --- | --- | --- |
| Logistic Regression | Tổ hợp tuyến tính các đặc trưng, qua hàm sigmoid ra xác suất | Đơn giản, dễ giải thích bằng hệ số |
| KNN | Dự đoán theo k hàng xóm gần nhất | Phi tham số, không giả định dạng quan hệ |
| Decision Tree | Chia dữ liệu bằng chuỗi câu hỏi có/không | Dễ đọc, bắt được tương tác giữa biến |
| Random Forest | Trung bình nhiều cây trên mẫu ngẫu nhiên | Giảm phương sai so với một cây |
| XGBoost | Các cây nối tiếp, cây sau sửa lỗi cây trước | Thường mạnh nhất trên dữ liệu bảng |
| ANN (MLP) | Mạng nơ-ron nhiều lớp ẩn | Học được quan hệ phi tuyến phức tạp |

Nếu máy chưa cài `xgboost`, mô hình XGBoost được bỏ qua.

In [ ]:
specs = candidate_specs()
pd.DataFrame(
    [{"mô hình": name, "lưới tham số": {k.replace("model__", ""): v for k, v in grid.items()}}
     for name, (_, grid) in specs.items()]
)

In [ ]:
best_params = {}
for name, (clf, grid) in specs.items():
    search = GridSearchCV(make_pipeline(clf), grid, scoring=SCORING, refit="roc_auc",
                          cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search.best_estimator_
    i = search.best_index_
    cv_rows.append({"model": name,
                    **{f"{m}_mean": search.cv_results_[f"mean_test_{m}"][i] for m in SCORING},
                    **{f"{m}_std": search.cv_results_[f"std_test_{m}"][i] for m in SCORING}})
    best_params[name] = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    print(f"{name:20s} ROC-AUC CV = {search.best_score_:.4f} | {best_params[name]}")

## 5. So sánh trên cross-validation

Bảng dưới là trung bình và độ lệch chuẩn qua 5 fold của tập huấn luyện. Thanh sai số trên biểu đồ là ±1 độ lệch chuẩn: hai mô hình có thanh chồng lên nhau thì **khác biệt không đáng kể**.

In [ ]:
cv_df = pd.DataFrame(cv_rows).set_index("model")
display(cv_df[[c for c in cv_df.columns if c.endswith("_mean")]].round(4))

plot_df = cv_df.drop("Baseline").sort_values("roc_auc_mean")
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh(plot_df.index, plot_df["roc_auc_mean"], xerr=plot_df["roc_auc_std"],
        color="#4C72B0", capsize=4)
ax.set_xlim(0.88, 0.96)
ax.set_xlabel("ROC-AUC trung bình 5-fold (± độ lệch chuẩn)")
ax.set_title("Các mô hình gần như ngang nhau trên cross-validation")
plt.tight_layout()
plt.show()

## 6. Đánh giá trên tập kiểm tra

Tập kiểm tra (1.700 dòng) chưa từng được dùng trong huấn luyện hay chọn tham số.

- **Accuracy:** tỉ lệ dự đoán đúng.
- **Precision:** trong số người bị cảnh báo, bao nhiêu người thật sự nợ ngủ.
- **Recall:** trong số người thật sự nợ ngủ, hệ thống bắt được bao nhiêu. Đây là chỉ số quan trọng nhất ở đề tài này, vì bỏ sót người có nguy cơ tệ hơn cảnh báo nhầm.
- **F1:** trung bình điều hòa của Precision và Recall.
- **ROC-AUC:** khả năng xếp người có nguy cơ cao hơn người không có nguy cơ, không phụ thuộc ngưỡng.

In [ ]:
rows, cms = [], []
for name, model in fitted.items():
    m, cm = _evaluate(name, model, X_test, y_test)
    rows.append(m)
    cms.append(cm)
test_df = pd.DataFrame(rows).set_index("model")
test_df.round(4)

In [ ]:
real = {k: v for k, v in fitted.items() if k != "Baseline"}

fig, ax = plt.subplots(figsize=(6.5, 6))
for name, model in real.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "--", color="grey", label="Đoán ngẫu nhiên")
ax.set_title("Đường ROC trên tập kiểm tra")
ax.legend(loc="lower right", fontsize=8)
plt.show()

In [ ]:
cols = 3
n = len(real)
fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(12, 3.6 * np.ceil(n / cols)))
axes = np.atleast_1d(axes).ravel()
for ax, (name, model) in zip(axes, real.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, display_labels=["Không", "Nợ ngủ"],
                                          cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name, fontsize=10)
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Chọn mô hình triển khai

Mô hình được chọn **chỉ dựa trên cross-validation của tập huấn luyện**, không dựa vào tập kiểm tra (để tránh chọn theo may rủi của tập kiểm tra).

Nhóm chọn **Logistic Regression** (`DEPLOY_MODEL` trong `src/train.py`) vì:

1. **Hiệu năng ngang mô hình tốt nhất:** chênh lệch ROC-AUC CV nhỏ hơn độ lệch chuẩn giữa các fold (ô dưới tính cụ thể).
2. **Dễ giải thích:** mỗi đặc trưng có một hệ số, cho biết tăng hay giảm nguy cơ và mạnh đến đâu (bước 9).
3. **Phù hợp lớp quyết định:** xác suất thay đổi mượt và đơn điệu theo số phút dùng máy, nên what-if và xếp hạng phương án cho kết quả nhất quán. Mô hình cây cho xác suất dạng bậc thang, giảm 5 phút có thể không đổi gì.

In [ ]:
best_cv = cv_df.drop("Baseline")["roc_auc_mean"].idxmax()
deployed_name = DEPLOY_MODEL if DEPLOY_MODEL in fitted else best_cv
gap = cv_df.loc[best_cv, "roc_auc_mean"] - cv_df.loc[deployed_name, "roc_auc_mean"]
std = cv_df.loc[deployed_name, "roc_auc_std"]

print(f"Mô hình có ROC-AUC CV cao nhất : {best_cv} ({cv_df.loc[best_cv, 'roc_auc_mean']:.4f})")
print(f"Mô hình triển khai             : {deployed_name} ({cv_df.loc[deployed_name, 'roc_auc_mean']:.4f})")
print(f"Chênh lệch = {gap:.4f}  |  độ lệch chuẩn giữa các fold = {std:.4f}")
print("=> Chênh lệch nhỏ hơn độ lệch chuẩn: hai mô hình ngang nhau về thống kê."
      if gap <= std else "=> Chênh lệch lớn hơn độ lệch chuẩn: cần cân nhắc lại lựa chọn.")
deployed = fitted[deployed_name]

## 8. Thí nghiệm rò rỉ dữ liệu

Nhãn `sleep_debt_category` được tính từ các chỉ số giấc ngủ (`total_sleep_hours`, `deep_sleep_pct`...). Nếu đưa các cột này vào đầu vào, mô hình gần như "chép đáp án". Hơn nữa, lúc ra quyết định buổi tối, người dùng **chưa biết** giấc ngủ đêm nay. Thí nghiệm dưới huấn luyện cùng một mô hình trên hai bộ đầu vào để thấy rõ khác biệt.

In [ ]:
from sklearn.linear_model import LogisticRegression

leak_rows = []
for leak in (False, True):
    a, b, c, d = prepare_data(include_leakage=leak, random_state=RANDOM_STATE)
    mdl = make_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced",
                                           random_state=RANDOM_STATE), include_leakage=leak)
    mdl.fit(a, c)
    m, _ = _evaluate("Logistic Regression", mdl, b, d)
    leak_rows.append({"đầu vào": "có cột giấc ngủ (rò rỉ)" if leak else "chỉ thói quen (đúng)",
                      **{k: v for k, v in m.items() if k != "model"}})
pd.DataFrame(leak_rows).set_index("đầu vào").round(4)

ROC-AUC gần 1,0 ở bản rò rỉ **không phải** mô hình tốt hơn, mà là mô hình gian lận. Bản đúng (chỉ dùng thói quen) mới phản ánh năng lực thật khi dùng trong thực tế.

## 9. Giải thích mô hình

**Hệ số Logistic Regression** (trên dữ liệu đã chuẩn hóa): hệ số dương làm tăng nguy cơ, hệ số âm làm giảm. `exp(hệ số)` là tỉ số odds khi đặc trưng tăng 1 độ lệch chuẩn (với biến số) hoặc khi thuộc nhóm đó (với biến phân loại).

In [ ]:
clf = deployed.named_steps["model"]
if hasattr(clf, "coef_"):
    coef = pd.DataFrame({
        "hệ số": clf.coef_[0],
        "tỉ số odds": np.exp(clf.coef_[0]),
    }, index=deployed.named_steps["prep"].get_feature_names_out()).sort_values("hệ số")
    display(coef.round(3))
else:
    print(f"{deployed_name} không có hệ số tuyến tính; xem độ quan trọng đặc trưng bên dưới.")

In [ ]:
fi = feature_importance(deployed, X_train, X_test, y_test)
top = fi.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top["feature"], top["importance"], color="#4C72B0")
ax.set_xlabel(top["method"].iloc[0])
ax.set_title(f"10 đặc trưng quan trọng nhất - {deployed_name}")
plt.tight_layout()
plt.show()

**Nhận xét:** nhịp sinh học và số phút dùng điện thoại trước khi ngủ là hai yếu tố mạnh nhất. Trong đó **số phút dùng máy là yếu tố mạnh nhất mà người dùng thay đổi được**, nên lớp quyết định lấy nó làm khuyến nghị chính. Các biến như độ sáng, bộ lọc ánh sáng xanh, tuổi gần như không ảnh hưởng.

## 10. Phân tích ngưỡng

Mô hình trả về xác suất; ngưỡng quyết định khi nào coi là "có nguy cơ". Ngưỡng thấp bắt được nhiều người có nguy cơ hơn (Recall cao) nhưng cảnh báo nhầm nhiều hơn (Precision thấp). Lớp quyết định dùng **30%** và **60%** để chia ba mức Thấp / Trung bình / Cao.

In [ ]:
thr = threshold_analysis(deployed, X_test, y_test)
display(thr.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
for col, label in [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]:
    ax.plot(thr["threshold"], thr[col], marker="o", label=label)
for t in (0.3, 0.6):
    ax.axvline(t, linestyle="--", color="grey")
ax.set_xlabel("Ngưỡng xác suất")
ax.set_ylabel("Giá trị")
ax.set_title("Hạ ngưỡng làm tăng Recall, đổi lại Precision giảm")
ax.legend()
plt.tight_layout()
plt.show()

**Ý nghĩa cho hệ thống:** ở ngưỡng 0,3, mô hình bắt được phần lớn người có nguy cơ (Recall cao nhất), phù hợp làm ranh giới "bắt đầu cần chú ý" (mức Trung bình). Ở ngưỡng 0,6, Precision cao, phù hợp làm ranh giới "chắc chắn có nguy cơ" (mức Cao). Đây là căn cứ cho hai ngưỡng của lớp quyết định.

## 11. Phân tích lỗi

Mô hình sai nhiều nhất ở nhóm nào? Bảng dưới tính tỉ lệ dự đoán sai theo nhịp sinh học, nghề nghiệp và khoảng số phút dùng máy.

In [ ]:
err = error_analysis(deployed, X_test, y_test)
err.round(3)

**Nhận xét:** mô hình sai nhiều nhất ở các nhóm "lưng chừng" (nhịp sinh học trung gian, dùng máy 31–90 phút), nơi tỉ lệ nợ ngủ thực tế gần 50%, tức là khó phân biệt nhất. Ở các nhóm rõ ràng (Cú đêm, dùng máy trên 120 phút) mô hình gần như không sai. Vì vậy hệ thống hiển thị **xác suất và mức nguy cơ**, không chỉ một kết luận có/không.

## 12. Hiệu chỉnh xác suất (calibration)

Ứng dụng hiển thị xác suất cho người dùng, nên cần kiểm tra xác suất có sát thực tế không: nếu mô hình nói 70%, thì khoảng 70% số người đó có thật sự nợ ngủ không. Đường càng gần đường chéo càng tốt.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
CalibrationDisplay.from_estimator(deployed, X_test, y_test, n_bins=10, ax=ax, name=deployed_name)
ax.set_title("Đường hiệu chỉnh xác suất")
plt.show()

**Nhận xét:** đường bám khá sát đường chéo ở vùng xác suất cao. Ở vùng thấp (dưới khoảng 40%), mô hình **hơi đánh giá thấp** nguy cơ: nhóm được dự đoán khoảng 15% thực tế có khoảng 24% người nợ ngủ. Nguyên nhân chính là `class_weight="balanced"` đẩy xác suất của lớp đa số (nợ ngủ, 60%) xuống.

Ảnh hưởng tới hệ thống: con số tuyệt đối ở mức Thấp nên đọc thận trọng, nhưng lớp quyết định dùng xác suất chủ yếu để **so sánh** trước và sau khi thay đổi thói quen, nên thứ tự các phương án không bị ảnh hưởng. Hướng cải thiện: hiệu chỉnh lại xác suất bằng `CalibratedClassifierCV`.

## 13. Chạy mô hình trên dữ liệu mới

Năm hồ sơ do nhóm tự tạo, **không có trong bộ dữ liệu**, đại diện cho các kiểu người dùng khác nhau. Mỗi hồ sơ được kiểm tra hợp lệ bằng `profile_to_frame()` rồi đưa vào mô hình.

In [ ]:
new_profiles = {
    "Sinh viên dùng TikTok 120 phút": dict(
        bedtime_phone_minutes=120, screen_brightness_pct=85, caffeine_post_5pm_mg=120,
        physical_activity_min=15, blue_light_filter_active=0, primary_bedtime_app="TikTok / Reels",
        age=21, gender="Male", occupation_type="Student", chronotype="Intermediate"),
    "Nhân viên văn phòng, thói quen tốt": dict(
        bedtime_phone_minutes=20, screen_brightness_pct=30, caffeine_post_5pm_mg=0,
        physical_activity_min=45, blue_light_filter_active=1, primary_bedtime_app="News / Reading",
        age=28, gender="Female", occupation_type="Corporate 9-to-5", chronotype="Morning Lark"),
    "Lập trình viên dùng YouTube 60 phút": dict(
        bedtime_phone_minutes=60, screen_brightness_pct=70, caffeine_post_5pm_mg=80,
        physical_activity_min=30, blue_light_filter_active=0, primary_bedtime_app="YouTube",
        age=26, gender="Male", occupation_type="Remote Tech", chronotype="Intermediate"),
    "Y tá làm ca, cú đêm": dict(
        bedtime_phone_minutes=95, screen_brightness_pct=75, caffeine_post_5pm_mg=160,
        physical_activity_min=20, blue_light_filter_active=0, primary_bedtime_app="YouTube",
        age=30, gender="Female", occupation_type="Healthcare / Shift Worker", chronotype="Night Owl"),
    "Freelancer dậy sớm, nhắn tin 40 phút": dict(
        bedtime_phone_minutes=40, screen_brightness_pct=50, caffeine_post_5pm_mg=0,
        physical_activity_min=60, blue_light_filter_active=1, primary_bedtime_app="Messaging / Chat",
        age=35, gender="Non-Binary", occupation_type="Freelance / Creative", chronotype="Morning Lark"),
}

new_X = pd.concat([profile_to_frame(p) for p in new_profiles.values()], ignore_index=True)
new_X.index = list(new_profiles.keys())
proba_new = deployed.predict_proba(new_X)[:, 1]

result = new_X[["bedtime_phone_minutes", "primary_bedtime_app", "chronotype", "occupation_type"]].copy()
result["xác suất nợ ngủ"] = proba_new.round(3)
result["mức nguy cơ"] = [risk_level(float(p)) for p in proba_new]
result

Từ con số dự đoán, **lớp quyết định** (`src/decision.py`) biến nó thành hành động. Ví dụ với hồ sơ sinh viên dùng TikTok 120 phút:

In [ ]:
profile = new_profiles["Sinh viên dùng TikTok 120 phút"]

print("Khuyến nghị:")
for r in recommend(profile):
    print(" -", r["hanh_dong"])

plans = rank_plans(profile, top_k=3)
pd.DataFrame([{
    "hạng": p["hang"],
    "phương án": p["phuong_an"],
    "nguy cơ trước": f"{p['truoc']:.1%}",
    "nguy cơ sau": f"{p['sau']:.1%} ({p['muc_sau']})",
    "điểm nỗ lực": p["no_luc"],
} for p in plans]).set_index("hạng")

Lưu ý: `recommend` và `rank_plans` dùng mô hình đã lưu trong `models/model.joblib` (mô hình ứng dụng đang chạy), còn bảng phía trên dùng mô hình vừa huấn luyện trong notebook. Ô ở bước 14 kiểm tra hai mô hình này cho cùng kết quả.

## 14. Lưu mô hình

Mặc định notebook **không ghi đè** mô hình chính thức. Bản chính thức được tạo bằng `python src/train.py`. Ô dưới kiểm tra mô hình vừa huấn luyện có khớp với `models/model.joblib` không; đặt `SAVE_MODEL = True` nếu muốn lưu bản mới từ notebook.

In [ ]:
SAVE_MODEL = False
MODEL_PATH = ROOT / "models" / "model.joblib"

if MODEL_PATH.exists():
    saved = joblib.load(MODEL_PATH)
    diff = np.abs(saved.predict_proba(X_test)[:, 1] - deployed.predict_proba(X_test)[:, 1]).max()
    print(f"Sai khác lớn nhất giữa mô hình notebook và models/model.joblib: {diff:.2e}")
    print("Khớp." if diff < 1e-6 else "Không khớp: hãy chạy lại python src/train.py.")
else:
    print("Chưa có models/model.joblib: chạy python src/train.py hoặc đặt SAVE_MODEL = True.")

if SAVE_MODEL:
    joblib.dump(deployed, MODEL_PATH)
    print(f"Đã lưu {deployed_name} vào {MODEL_PATH}")

## Kết luận

- Cả 6 mô hình đều vượt xa baseline (ROC-AUC khoảng 0,92–0,94 so với 0,50); các mô hình mạnh nhất gần như ngang nhau.
- Nhóm triển khai **Logistic Regression** vì hiệu năng ngang mô hình tốt nhất, dễ giải thích, và cho xác suất mượt phù hợp với what-if.
- Thí nghiệm rò rỉ cho thấy loại bỏ các cột kết quả giấc ngủ là bắt buộc.
- Số phút dùng điện thoại là yếu tố mạnh nhất mà người dùng thay đổi được, là cơ sở cho khuyến nghị chính của hệ thống.
- **Hạn chế:** dữ liệu có dấu hiệu mô phỏng và là dữ liệu cắt ngang, nên kết quả minh họa phương pháp, không chứng minh quan hệ nhân quả.